# 🍷 Wine Quality Prediction using XGBoost
Predicting wine quality score (3–8) and classifying as Good/Bad using the UCI Wine Quality dataset.

**Dataset:** Red + White Wine combined  
**Model:** XGBoost Classifier  
**Target:** Quality score & Good/Bad classification

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, classification_report,
    confusion_matrix, mean_absolute_error, r2_score
)
from xgboost import XGBClassifier
import pickle

print('Libraries imported successfully!')

## 2. Load Dataset

In [ ]:
# Load both datasets
red = pd.read_csv('https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-red.csv', sep=';')
white = pd.read_csv('https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-white.csv', sep=';')

# Add wine type column
red['wine_type'] = 0    # 0 = Red
white['wine_type'] = 1  # 1 = White

# Combine
df = pd.concat([red, white], ignore_index=True)

print(f'Total samples: {len(df)}')
print(f'Red wine: {len(red)} | White wine: {len(white)}')
df.head()

## 3. Exploratory Data Analysis

In [ ]:
print('Shape:', df.shape)
print('\nMissing Values:')
print(df.isnull().sum())
print('\nBasic Statistics:')
df.describe()

In [ ]:
# Quality distribution
plt.figure(figsize=(8, 4))
df['quality'].value_counts().sort_index().plot(kind='bar', color='steelblue', edgecolor='black')
plt.title('Wine Quality Distribution')
plt.xlabel('Quality Score')
plt.ylabel('Count')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap
plt.figure(figsize=(12, 8))
sns.heatmap(df.corr(), annot=True, fmt='.2f', cmap='coolwarm', linewidths=0.5)
plt.title('Feature Correlation Heatmap')
plt.tight_layout()
plt.show()

## 4. Feature Engineering

In [ ]:
# Create binary classification target: Good (1) if quality >= 7, else Bad (0)
df['good_wine'] = (df['quality'] >= 7).astype(int)

print('Good/Bad wine distribution:')
print(df['good_wine'].value_counts())
print(f"\nGood wine %: {df['good_wine'].mean()*100:.1f}%")

In [ ]:
# Features and targets
FEATURES = [
    'fixed acidity', 'volatile acidity', 'citric acid', 'residual sugar',
    'chlorides', 'free sulfur dioxide', 'total sulfur dioxide', 'density',
    'pH', 'sulphates', 'alcohol', 'wine_type'
]

X = df[FEATURES]
y = df['good_wine']  # Binary classification: Good (1) or Bad (0)

print('Features:', FEATURES)
print('Target distribution:')
print(y.value_counts())

## 5. Train-Test Split & Scaling

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f'Train size: {X_train.shape[0]}')
print(f'Test size:  {X_test.shape[0]}')

## 6. Train XGBoost Model

In [ ]:
model = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    use_label_encoder=False,
    eval_metric='logloss',
    random_state=42
)

model.fit(X_train_scaled, y_train)
print('Model trained successfully!')

## 7. Model Evaluation

In [ ]:
y_pred = model.predict(X_test_scaled)

acc = accuracy_score(y_test, y_pred)
print(f'Accuracy: {acc*100:.2f}%')
print('\nClassification Report:')
print(classification_report(y_test, y_pred, target_names=['Bad Wine', 'Good Wine']))

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Bad Wine', 'Good Wine'],
            yticklabels=['Bad Wine', 'Good Wine'])
plt.title('Confusion Matrix')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.tight_layout()
plt.show()

In [ ]:
# Feature Importance
importance = pd.Series(model.feature_importances_, index=FEATURES).sort_values(ascending=True)

plt.figure(figsize=(8, 6))
importance.plot(kind='barh', color='steelblue', edgecolor='black')
plt.title('XGBoost Feature Importance')
plt.xlabel('Importance Score')
plt.tight_layout()
plt.show()

## 8. Save Model & Scaler

In [ ]:
import os
os.makedirs('pkl', exist_ok=True)

with open('pkl/wine_model.pkl', 'wb') as f:
    pickle.dump(model, f)

with open('pkl/wine_scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

print('Model and scaler saved to pkl/ folder!')
print('Files created:')
print('  - pkl/wine_model.pkl')
print('  - pkl/wine_scaler.pkl')

## 9. Test Prediction

In [ ]:
# Sample prediction
sample = pd.DataFrame([{
    'fixed acidity': 7.4,
    'volatile acidity': 0.70,
    'citric acid': 0.00,
    'residual sugar': 1.9,
    'chlorides': 0.076,
    'free sulfur dioxide': 11.0,
    'total sulfur dioxide': 34.0,
    'density': 0.9978,
    'pH': 3.51,
    'sulphates': 0.56,
    'alcohol': 9.4,
    'wine_type': 0  # Red wine
}])

sample_scaled = scaler.transform(sample)
pred = model.predict(sample_scaled)[0]
prob = model.predict_proba(sample_scaled)[0]

print(f'Prediction: {"Good Wine 🍷" if pred == 1 else "Bad Wine"}')
print(f'Probability - Bad: {prob[0]*100:.1f}% | Good: {prob[1]*100:.1f}%')